# 04b — Test congelado: YOLO11n-seg vs. PatchCore en FabriD v3

Este notebook **no entrena ni elige parámetros**. Carga exclusivamente los artefactos de la [validación completa, versión 3 de Kaggle](https://www.kaggle.com/code/victoriaprsico/04-comparacion-fabrid?scriptVersionId=372676240), verifica su coherencia y evalúa una sola vez el split test común. Las dos máscaras usan la geometría del export anotado. El bootstrap por `source_id` es pareado entre métodos.

**Antes de ejecutar:** descargar `results_fabrid04.zip` de la versión 3 Successful de Kaggle, crear un dataset privado con ese ZIP y adjuntarlo como Input, además de `fabrid-v3-export-anotado`. Verificar en el editor que ambos Input aparecen. El ZIP debe contener `results/fabrid/comparacion04/`, incluido `configuracion.json`, `fabrid_comparison.py`, `metricas_valid.json`, `patchcore_memoria.npz` y `yolo11n_seg/weights/best.pt`.

La tabla de validación ya fijó los umbrales: YOLO **0.16507** y PatchCore **2.669366**. Se leerán los valores de precisión completa de `metricas_valid.json`; los números redondeados sólo sirven como control. `ABRIR_TEST` comienza en `False`.

In [ ]:
# Revisar los Input y ejecutar primero con ABRIR_TEST=False.
# Con los controles de validación aprobados, activar True y usar Save Version → Save & Run All UNA vez.
ABRIR_TEST = False
SEMILLA_BOOTSTRAP = 20261122
REPLICAS_BOOTSTRAP = 2000
RUTA_EXPORT = ""      # Detección automática si queda vacío
RUTA_ARTEFACTOS = ""  # Detección automática del ZIP de validación si queda vacío


## 1. Entorno y recuperación de los artefactos de validación

In [ ]:
import sys, subprocess, os, json, hashlib, zipfile, shutil, time
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
from PIL import Image

WORK = Path("/kaggle/working")
if not WORK.is_dir():
    raise RuntimeError("Este notebook está preparado para Kaggle.")
REPO = WORK / "tp_final_vision_04b"
COMMIT = "3bda925932d0d42cedb49857993e8f1ba28df75d"
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/victoriapersico/tp_final_vision.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215",
                "sahi==0.14.0", "pycocotools==2.0.11", "scipy", "scikit-learn", "pandas"], check=True)
sys.path.insert(0, str(REPO / "src"))
import anomaly_baseline as ab
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
# Kaggle puede montar el ZIP tal cual o extraer su contenido al crear el dataset.
if RUTA_ARTEFACTOS:
    roots = [Path(RUTA_ARTEFACTOS)]
else:
    roots = list(INPUT.rglob("configuracion.json"))
    for z in INPUT.rglob("*.zip"):
        with zipfile.ZipFile(z) as archive:
            names = archive.namelist()
            if "results/fabrid/comparacion04/configuracion.json" not in names:
                continue
            dest = Path("/tmp/fabrid04_valid")
            if not dest.exists():
                archive.extractall(dest)
            roots.append(dest / "results/fabrid/comparacion04/configuracion.json")
roots = sorted({(p if p.is_dir() else p.parent).resolve() for p in roots
                if (p if p.is_dir() else p.parent).name == "comparacion04"})
if len(roots) != 1:
    raise RuntimeError(f"Se esperaba UNA carpeta comparacion04 de validación; encontradas: {roots}")
VAL = roots[0]
required = ["configuracion.json", "metricas_valid.json", "fabrid_comparison.py",
            "patchcore_memoria.npz", "patchcore_completado.json", "yolo11n_seg/completado.json",
            "yolo11n_seg/weights/best.pt"]
missing = [p for p in required if not (VAL / p).is_file()]
if missing:
    raise FileNotFoundError(f"Faltan artefactos de la versión 3: {missing}")
config = json.loads((VAL / "configuracion.json").read_text())
metrics_valid = json.loads((VAL / "metricas_valid.json").read_text())
if config["modo_prueba"] is not False or config["evaluar_test"] is not False:
    raise ValueError("Los artefactos no proceden de validación completa con test cerrado")
if config["commit"] != COMMIT or config["semilla"] != 0:
    raise ValueError("El commit o la semilla difieren de la validación prevista")
if set(metrics_valid) != {"YOLO11n-seg", "PatchCore"}:
    raise ValueError("Faltan métricas de alguno de los dos métodos")
thresholds = {k: float(metrics_valid[k]["umbral_valid"]) for k in metrics_valid}
for name, approx in (("YOLO11n-seg", .16507), ("PatchCore", 2.669366)):
    m = metrics_valid[name]
    if m["modo_prueba"] is not False or m["split"] != "valid" or m["imagenes"] != 375:
        raise ValueError("Se requieren las métricas de validación completa")
    if not np.isclose(thresholds[name], approx, atol=0.00002):
        raise ValueError(f"Umbral inesperado para {name}: {thresholds[name]}")
if json.loads((VAL / "yolo11n_seg/completado.json").read_text()) != config["supervisado"]:
    raise ValueError("La firma de YOLO no coincide con la configuración guardada")
if json.loads((VAL / "patchcore_completado.json").read_text())["backbone"] != config["pc"]["backbone"]:
    raise ValueError("La firma de PatchCore no coincide con la configuración guardada")
spec = importlib.util.spec_from_file_location("fabrid_comparison", VAL / "fabrid_comparison.py")
fc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fc)
print("Artefactos de validación:", VAL)
print("Umbrales congelados de precisión completa:", thresholds)


## 2. Manifiesto y verificación en validación, sin mirar test

In [ ]:
def es_export(path):
    return all((path / s / k).is_dir()
               for s in ("train", "valid", "test") for k in ("images", "labels"))
if RUTA_EXPORT:
    export = Path(RUTA_EXPORT)
else:
    candidates = sorted({p.parent for p in INPUT.rglob("train") if p.is_dir() and es_export(p.parent)})
    if len(candidates) != 1:
        raise RuntimeError(f"Se esperaba UN export anotado FabriD: {candidates}")
    export = candidates[0]
if not es_export(export):
    raise FileNotFoundError("El export requiere train/valid/test con images/labels")
DATA = Path("/tmp/rewear_fabrid04_test")
link = DATA / "fabrid_v3/annotations_export"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() and link.resolve() != export.resolve():
    raise ValueError("Input FabriD cambiado en esta sesión")
if not link.exists():
    link.symlink_to(export.resolve(), target_is_directory=True)
os.environ["REWEAR_DATA_DIR"] = str(DATA)
manifest = REPO / "reports/fabrid/fab4_split_agrupado.csv"
if hashlib.sha256(manifest.read_bytes()).hexdigest() != config["manifiesto_sha256"]:
    raise ValueError("El manifiesto cambió respecto de la validación")
items = fc.load_items(DATA, manifest)
splits = {s: [i for i in items if i["split"] == s] for s in ("train", "valid", "test")}
for split, normal, defective in (("train", 429, 1221), ("valid", 93, 282), ("test", 85, 278)):
    rows = splits[split]
    if (sum(i["label"] == 0 for i in rows), sum(i["label"] == 1 for i in rows)) != (normal, defective):
        raise ValueError(f"Conteo inesperado en {split}")
if set(i["source_id"] for i in splits["train"]) & set(i["source_id"] for i in splits["valid"] + splits["test"]):
    raise ValueError("Fuga de fuentes entre train y evaluación")
if set(i["source_id"] for i in splits["valid"]) & set(i["source_id"] for i in splits["test"]):
    raise ValueError("Fuga de fuentes entre valid y test")
print("Splits auditados:", {k: len(v) for k,v in splits.items()})
pc_signature = json.loads((VAL / "patchcore_completado.json").read_text())
expected_signature = dict(config["pc"], semilla=config["semilla"],
                          train=[i["id"] for i in splits["train"] if i["label"] == 0])
if pc_signature != expected_signature:
    raise ValueError("El banco PatchCore no corresponde a las normales de train del manifiesto")

sup = YOLO(str(VAL / "yolo11n_seg/weights/best.pt"))
pc = ab.PatchCore(backbone=config["pc"]["backbone"], size=config["pc"]["size"],
                  coreset=config["pc"]["coreset"], seed=config["semilla"])
with np.load(VAL / "patchcore_memoria.npz") as saved:
    pc.state = {key: saved[key] for key in saved.files}
# Una predicción en validación por método comprueba la carga, la geometría y el evaluador.
sample = splits["valid"][:1]
for name, predicted in (("YOLO11n-seg", fc.predict_yolo(sup, sample, imgsz=config["supervisado"]["imgsz"])),
                        ("PatchCore", fc.predict_patchcore(pc, sample))):
    fc.validate_maps(sample, predicted)
    if not np.isfinite(predicted[0]).all():
        raise ValueError(f"Puntajes no finitos: {name}")
    print("Carga e inferencia en valid comprobadas:", name, predicted[0].shape)
print("Test sigue cerrado:", not ABRIR_TEST)


## 3. Evaluación única en test

Esta celda sólo corre con `ABRIR_TEST=True`. Reutiliza los umbrales globales de validación. No calcula umbrales ni entrena con test. Los 2000 remuestreos pareados por `source_id` calculan IC percentiles del 95 % para AUROC imagen y diferencia YOLO menos PatchCore; las otras métricas se reportan como estimaciones puntuales.

In [ ]:
if not ABRIR_TEST:
    print("Test cerrado. Revisar controles anteriores y ejecutar una versión nueva con ABRIR_TEST=True.")
else:
    test = splits["test"]
    assert len(test) == 363 and sum(i["label"] == 0 for i in test) == 85
    OUT = WORK / "results/fabrid/comparacion04_test"
    OUT.mkdir(parents=True, exist_ok=True)
    recorded = {"commit": COMMIT, "valid_config_sha256": hashlib.sha256((VAL / "configuracion.json").read_bytes()).hexdigest(),
                "valid_metrics_sha256": hashlib.sha256((VAL / "metricas_valid.json").read_bytes()).hexdigest(),
                "manifest_sha256": config["manifiesto_sha256"], "umbrales_valid": thresholds,
                "semilla_bootstrap": SEMILLA_BOOTSTRAP, "replicas_bootstrap": REPLICAS_BOOTSTRAP,
                "dataset_export": str(export)}
    (OUT / "protocolo_congelado.json").write_text(json.dumps(recorded, ensure_ascii=False, indent=2))
    resultados, image_scores = {}, {}
    for name, predictor in (("YOLO11n-seg", lambda: fc.predict_yolo(sup, test, imgsz=config["supervisado"]["imgsz"])),
                            ("PatchCore", lambda: fc.predict_patchcore(pc, test))):
        start = time.perf_counter()
        maps = predictor()
        elapsed = (time.perf_counter() - start) / len(test)
        m = fc.evaluate_maps(test, maps, thresholds[name], confirm_test=True)
        m["segundos_por_imagen"] = elapsed
        resultados[name] = m
        image_scores[name] = np.asarray([float(x.max()) for x in maps])
        folder = OUT / name.replace("-", "_")
        folder.mkdir(exist_ok=True)
        np.savez_compressed(folder / "mapas_test.npz", **{f"map_{j:04d}": x for j,x in enumerate(maps)})
        (folder / "orden_imagenes.json").write_text(json.dumps([i["id"] for i in test]))
        (folder / "metricas_test.json").write_text(json.dumps(m, ensure_ascii=False, indent=2))
        del maps
    display(pd.DataFrame(resultados).T)
    (OUT / "metricas_test.json").write_text(json.dumps(resultados, ensure_ascii=False, indent=2))

    # Repetir la misma selección de fuentes en ambos modelos preserva el pareamiento.
    from sklearn.metrics import roc_auc_score
    groups = sorted({i["source_id"] for i in test})
    indexes = {g: np.asarray([j for j,i in enumerate(test) if i["source_id"] == g]) for g in groups}
    labels = np.asarray([i["label"] for i in test])
    rng = np.random.default_rng(SEMILLA_BOOTSTRAP)
    draws = []
    for _ in range(REPLICAS_BOOTSTRAP):
        chosen = rng.choice(groups, len(groups), replace=True)
        ix = np.concatenate([indexes[g] for g in chosen])
        if len(np.unique(labels[ix])) < 2:
            continue
        a = roc_auc_score(labels[ix], image_scores["YOLO11n-seg"][ix])
        b = roc_auc_score(labels[ix], image_scores["PatchCore"][ix])
        draws.append((a, b, a-b))
    if len(draws) < int(.95 * REPLICAS_BOOTSTRAP):
        raise ValueError("Demasiadas muestras bootstrap sin ambas clases")
    ci = np.percentile(draws, [2.5, 97.5], axis=0)
    intervalos = {name: {"estimacion": value, "IC95": ci[:,j].tolist()}
                  for j,(name,value) in enumerate((("YOLO11n-seg AUROC_imagen", resultados["YOLO11n-seg"]["AUROC_imagen"]),
                                                   ("PatchCore AUROC_imagen", resultados["PatchCore"]["AUROC_imagen"]),
                                                   ("Diferencia AUROC_imagen YOLO-PatchCore", resultados["YOLO11n-seg"]["AUROC_imagen"] - resultados["PatchCore"]["AUROC_imagen"]))) }
    (OUT / "intervalos_fuente.json").write_text(json.dumps(intervalos, ensure_ascii=False, indent=2))
    display(pd.DataFrame(intervalos).T)
    print("Test evaluado una vez con modelos y umbrales de validación congelados.")


## 4. Guardar resultados

El ZIP de test contiene mapas, métricas, intervalos y el protocolo congelado. Los pesos y el banco siguen en el Input de validación: conservar ambos datasets/versiones Kaggle para reproducir este test. No cambiar parámetros según los resultados de test.

In [ ]:
if ABRIR_TEST:
    zip_path = shutil.make_archive(str(WORK / "results_fabrid04_test"), "zip", root_dir=WORK,
                                   base_dir="results/fabrid/comparacion04_test")
    print("Archivo para descargar en Output:", zip_path)
else:
    print("Sin ZIP de test: ABRIR_TEST=False.")
